In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms


class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, shuffle=False)


model = Net()
optimizer = optim.Adadelta(model.parameters(), lr=1.0)

for epoch in range(3):
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch + 1, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
correct = 0
total = 0
with torch.no_grad():
    for data, target in test_loader:
        output = model(data)
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

print('Accuracy of the network on the 10000 test images: %d %%' % (
    100 * correct / total))

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 488kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.59MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.26MB/s]


Train Epoch: 1 [0/60000 (0%)]	Loss: 2.306652
Train Epoch: 1 [6400/60000 (11%)]	Loss: 0.124795
Train Epoch: 1 [12800/60000 (21%)]	Loss: 0.147918
Train Epoch: 1 [19200/60000 (32%)]	Loss: 0.085109
Train Epoch: 1 [25600/60000 (43%)]	Loss: 0.085829
Train Epoch: 1 [32000/60000 (53%)]	Loss: 0.032530
Train Epoch: 1 [38400/60000 (64%)]	Loss: 0.145679
Train Epoch: 1 [44800/60000 (75%)]	Loss: 0.067166
Train Epoch: 1 [51200/60000 (85%)]	Loss: 0.060847
Train Epoch: 1 [57600/60000 (96%)]	Loss: 0.054749
Train Epoch: 2 [0/60000 (0%)]	Loss: 0.095920
Train Epoch: 2 [6400/60000 (11%)]	Loss: 0.009823
Train Epoch: 2 [12800/60000 (21%)]	Loss: 0.061832
Train Epoch: 2 [19200/60000 (32%)]	Loss: 0.044227
Train Epoch: 2 [25600/60000 (43%)]	Loss: 0.267176
Train Epoch: 2 [32000/60000 (53%)]	Loss: 0.009809
Train Epoch: 2 [38400/60000 (64%)]	Loss: 0.139539
Train Epoch: 2 [44800/60000 (75%)]	Loss: 0.043329
Train Epoch: 2 [51200/60000 (85%)]	Loss: 0.029961
Train Epoch: 2 [57600/60000 (96%)]	Loss: 0.027155
Train Epoch: